# Social Engine Users — Data Cleaning & EDA

This notebook documents the full process of cleaning the raw `Social_Engine_Users_raw.csv`
export and preparing it for analysis. Every cleaning decision is explained in a markdown
cell immediately above the code that implements it, under **Assumption**.

**Dataset columns**
| Column | Description |
|---|---|
| `user_id` | Unique identifier for each user |
| `location` | City, Country of the user |
| `language` | ISO-639-1 language code of the user's account |
| `account_created` | Date the account was created |
| `follower_count` | Number of followers on the account |


## 1. Setup & Load Raw Data

In [ ]:
import pandas as pd
import numpy as np

raw = pd.read_csv('../data/Social_Engine_Users_raw.csv')
print("Raw shape:", raw.shape)
raw.head()

Raw shape: (1560, 5)


## 2. Initial Data Quality Audit

Before cleaning, profile the raw data to see what actually needs fixing.

In [ ]:
print("Null counts:\n", raw.isnull().sum())
print("\nExact duplicate rows:", raw.duplicated().sum())
print("\nDuplicate user_id count:", raw['user_id'].duplicated().sum())
print("\nUnique language values:", sorted(raw['language'].dropna().unique()))
print("\naccount_created sample values:", raw['account_created'].sample(8, random_state=1).tolist())
print("\nfollower_count dtype:", raw['follower_count'].dtype)

## 3. Remove Exact Duplicate Rows

**Assumption:** Fully identical rows (every field matches) are export artifacts — e.g. the
same record was pulled twice during data extraction — not genuine repeat events. They carry
no additional information, so they are safely dropped.

In [ ]:
before = len(raw)
df = raw.drop_duplicates()
print(f"Dropped {before - len(df)} exact duplicate rows -> {len(df)} rows remain")

## 4. Standardize Text Fields (`location`, `language`)

**Assumption:** Casing and stray whitespace (e.g. `"BERLIN, GERMANY"`, `"  Berlin, Germany  "`,
`"berlin, germany"`) are formatting noise from inconsistent input sources, not distinct
categories. They are normalized to a single consistent representation (Title Case for
location, lowercase for language) so that grouping/aggregation isn't fragmented across
near-duplicate labels.

In [ ]:
df['location'] = df['location'].str.strip()
df['language'] = df['language'].str.strip().str.lower()

df['location'] = df['location'].apply(lambda x: x.title() if isinstance(x, str) else x)

# Title-casing breaks 3-letter country acronyms (Uae -> UAE); restore them
acronym_fix = {'Uae': 'UAE', 'Uk': 'UK', 'Usa': 'USA'}
for wrong, right in acronym_fix.items():
    df['location'] = df['location'].str.replace(rf'\b{wrong}\b', right, regex=True)

print(df['location'].unique()[:10])

## 5. Fix Language Code Typos

**Assumption:** Malformed codes such as `"enn"` are single-character data-entry duplications
of a valid 2-letter ISO code (`"en"` + trailing repeat of the last letter). Any code that
isn't in the known set of 10 valid language codes, but matches this doubled-letter pattern,
is corrected to its 2-letter root. Codes that still don't resolve would be flagged for manual
review (none remained in this dataset).

In [ ]:
valid_langs = {'en','fr','de','es','zh','ja','ru','ar','pt','hi'}

def fix_lang(x):
    if pd.isna(x):
        return x
    if x in valid_langs:
        return x
    if len(x) == 3 and x[:2] in valid_langs:
        return x[:2]
    return x

df['language'] = df['language'].apply(fix_lang)
unresolved = sorted(set(df['language'].dropna()) - valid_langs)
print("Unresolved/invalid language codes remaining:", unresolved)

## 6. Parse Mixed Date Formats

**Assumption:** `account_created` was exported with at least four different date formats
mixed together (`YYYY-MM-DD`, `DD/MM/YYYY`, `MM-DD-YYYY`, `DD-Mon-YYYY`) — a common symptom
of data being merged from multiple source systems/locales. Each value is parsed against the
known format list and standardized to ISO 8601 (`YYYY-MM-DD`) for consistent, unambiguous
downstream analysis.

In [ ]:
def parse_date(d):
    if pd.isna(d):
        return pd.NaT
    for fmt in ('%Y-%m-%d', '%d/%m/%Y', '%m-%d-%Y', '%d-%b-%Y'):
        try:
            return pd.to_datetime(d, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.to_datetime(d, errors='coerce')

df['account_created'] = df['account_created'].apply(parse_date)
print("Unparseable dates after conversion:", df['account_created'].isna().sum())
print("Date range:", df['account_created'].min(), "to", df['account_created'].max())

## 7. Clean `follower_count`

**Assumptions:**
- Non-numeric placeholder values (e.g. `"N/A"`) represent missing measurements and are
  coerced to `NaN` rather than dropped outright at this stage — they're handled together with
  other missing values in Section 8.
- Negative follower counts are impossible in reality; they're treated as a **sign error**
  (e.g. a stray minus sign from a broken export/parsing step) and corrected by taking the
  absolute value, rather than discarded, since the magnitude still looks plausible.
- Values far above the IQR upper fence are treated as **data-entry magnitude errors** (e.g.
  an extra trailing zero) rather than genuinely viral accounts, since this is a general user
  base sample, not an influencer dataset. They are capped (winsorized) at the upper fence
  rather than deleted, to avoid losing otherwise-valid rows.

In [ ]:
df['follower_count'] = pd.to_numeric(df['follower_count'], errors='coerce')

neg_count = (df['follower_count'] < 0).sum()
df.loc[df['follower_count'] < 0, 'follower_count'] = df.loc[df['follower_count'] < 0, 'follower_count'].abs()
print(f"Fixed {neg_count} negative values via absolute value")

q1, q3 = df['follower_count'].quantile(0.25), df['follower_count'].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr
outlier_count = (df['follower_count'] > upper_bound).sum()
df.loc[df['follower_count'] > upper_bound, 'follower_count'] = upper_bound
print(f"IQR upper bound: {upper_bound:.0f} — capped {outlier_count} high-end outliers")

## 8. Handle Missing Values

**Assumptions:**
- **`location`** — dropped when missing. Location underpins the geographic analysis in this
  report and cannot be reliably inferred from the other fields, so rows without it are
  excluded rather than imputed with a placeholder that would distort geographic breakdowns.
- **`language`** — imputed with the **mode** (most frequent language). Missing-at-random
  language values are unlikely to correlate strongly with a specific language, so the mode is
  a low-bias default that preserves the row for the rest of the analysis.
- **`follower_count`** — imputed with the **median** rather than mean, since the distribution
  is right-skewed and the median is more robust to the outliers already capped in Section 7.

In [ ]:
print("Missing values before imputation:")
print(df.isnull().sum())

before = len(df)
df = df.dropna(subset=['location'])
print(f"\nDropped {before - len(df)} rows missing location")

lang_mode = df['language'].mode()[0]
df['language'] = df['language'].fillna(lang_mode)
print(f"Imputed missing language with mode: '{lang_mode}'")

fc_median = df['follower_count'].median()
df['follower_count'] = df['follower_count'].fillna(fc_median).round().astype(int)
print(f"Imputed missing follower_count with median: {fc_median:.0f}")

## 9. Resolve Duplicate `user_id`s

**Assumption:** `user_id` should be unique — any repeats left after Section 3 (i.e. same ID,
but at least one differing field) are treated as conflicting records from the same user,
most likely caused by a profile update being captured twice during export. The **first
occurrence** is kept, on the assumption the raw export is roughly chronological and the
first record is closest to the original registration snapshot this dataset represents.

In [ ]:
before = len(df)
df = df.drop_duplicates(subset='user_id', keep='first')
print(f"Dropped {before - len(df)} duplicate user_id rows (kept first occurrence)")

## 10. Final Formatting & Export

In [ ]:
df['account_created'] = df['account_created'].dt.strftime('%Y-%m-%d')
df = df.reset_index(drop=True)

print("Final cleaned shape:", df.shape)
print(df.isnull().sum())
df.to_csv('../data/Social_Engine_Users_cleaned.csv', index=False)
df.head()

Final cleaned shape: (1474, 5)


## 11. Cleaning Summary

| Step | Rows affected |
|---|---|
| Exact duplicate rows removed | 9 |
| Missing `location` dropped | 30 |
| Missing `language` imputed (mode) | 30 |
| Missing `follower_count` imputed (median) | 45 |
| Negative `follower_count` corrected | 22 |
| High-end outliers capped (IQR) | 11 |
| Duplicate `user_id` rows resolved | 47 |
| **Raw rows → Cleaned rows** | **1560 → 1474** |

The cleaned dataset is saved to `../data/Social_Engine_Users_cleaned.csv` and used as the
input for the EDA report in `../reports/EDA_Report.md`.

## 12. Quick Visual Sanity Check

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12,4))
sns.histplot(df['follower_count'], bins=30, kde=True, ax=axes[0])
axes[0].set_title('Follower Count Distribution (cleaned)')
df['language'].value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('Users per Language (cleaned)')
plt.tight_layout()
plt.show()